In [6]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.interpolate import interp1d
from scipy.stats import mode
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
import os

In [3]:
def load_wesad_data(directory):
    data = {}
    for participant_id in os.listdir(directory):
        participant_path = os.path.join(directory, participant_id)
        if os.path.isdir(participant_path):
            participant_data = {}
            for file_name in os.listdir(participant_path):
                file_path = os.path.join(participant_path, file_name)
                if file_name.endswith('.pkl'):
                    participant_data[file_name.split('.')[0]] = pd.read_pickle(file_path)
            data[participant_id] = participant_data
    return data

wesad_data = load_wesad_data('C:/Users/rusha/Desktop/Uni_Freiburg_Notes/MDD/WESAD')

In [5]:
def extract_and_process_data_chest(subject_id):
    subject_data = wesad_data[subject_id]
    subject_details = subject_data[subject_id]
    chest_data = subject_details['signal']['chest']
    labels = subject_details['label']
    
    # flatten and reshape each signal
    ecg = chest_data['ECG'].flatten().reshape(-1, 1)
    emg = chest_data['EMG'].flatten().reshape(-1, 1)
    eda = chest_data['EDA'].flatten().reshape(-1, 1)
    temp = chest_data['Temp'].flatten().reshape(-1, 1)
    resp = chest_data['Resp'].flatten().reshape(-1, 1)
    
    # # print the shapes of all features
    # print(f"ECG shape: {ecg.shape}")
    # print(f"EMG shape: {emg.shape}")
    # print(f"EDA shape: {eda.shape}")
    # print(f"TEMP shape: {temp.shape}")
    # print(f"RESP shape: {resp.shape}")
    
    # combine the features into a single array
    features = np.hstack((
        ecg,
        emg,
        eda,
        temp,
        resp
    ))

    return features, labels

# extract and process data from subjects S10 and S11
features_chest_s2, labels_chest_s2 = extract_and_process_data_chest('S2')
features_chest_s15, labels_chest_s15 = extract_and_process_data_chest('S15')
features_chest_s16, labels_chest_s16 = extract_and_process_data_chest('S16')
# features_chest_s17, labels_chest_s17 = extract_and_process_data_chest('S17')

# combine features and labels from both subjects
combined_features = np.vstack((features_chest_s2))
combined_labels = np.hstack((labels_chest_s2))

In [9]:
l = pd.DataFrame(np.hstack((combined_features, combined_labels.reshape(-1, 1))))
l = l[l[5].isin([1.0, 2.0, 3.0])]
chest_combined_features = l.loc[:,:4]
chest_combined_labels = l.loc[:, 5]
loo = LeaveOneOut()

# Initialize variables to store the predictions and the ground truth labels
y_true = []
y_pred = []

# Initialize the Random Forest Classifier
clf_chest = DecisionTreeClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# Loop over all the samples in the dataset
for train_index, test_index in loo.split(chest_combined_features):
    # Split the data into training and test sets for this iteration
    X_train, X_test = chest_combined_features.iloc[train_index], chest_combined_features.iloc[test_index]
    y_train, y_test = chest_combined_labels.iloc[train_index], chest_combined_labels.iloc[test_index]
    
    # Normalize the data (use StandardScaler for each fold)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # Train the model on the training set
    clf_chest.fit(X_train, y_train)
    
    # Predict the label for the test set
    y_pred_single = clf_chest.predict(X_test)
    
    # Store the prediction and the true label
    y_true.append(y_test.iloc[0])

    y_pred.append(y_pred_single[0])

# Convert lists to numpy arrays for evaluation
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the overall accuracy
loo_accuracy = accuracy_score(y_true, y_pred)
print(f"LOOCV Accuracy: {loo_accuracy}")


loo_report = classification_report(y_true, y_pred)
print("LOOCV Classification Report:")
print(loo_report)